# Build datasets — vLLM generation (Colab)

Expands the committed seeds into the full structural dataset (100 instructions × 4 categories = 400)
and conflict-pairs dataset (200 pairs) by prompting an HF instruct model served via vLLM.

**Runtime**: GPU (A100 recommended; T4 works for models ≤ 7B). **Not Colab-free-CPU.**

Output lands in `Mech_spoof/data/` — copy to your Drive or download the JSONs at the end.


In [ ]:
# Install vLLM (nightly, Gemma 4 needs it) + transformers 5.5.0 + clone the repo
!pip install -q vllm 
![ -d /content/Mech_spoof ] || git clone https://github.com/ChuloIva/Mech_spoof.git /content/Mech_spoof
%cd /content/Mech_spoof
!pip install -q -e .

In [ ]:
# HF sign-in — paste your token when prompted (input is hidden).
# Get one at https://huggingface.co/settings/tokens (read access is enough for public models;
# use a token with access granted for gated models like Gemma/Llama).
import os, getpass
from huggingface_hub import login

token = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ').strip()
os.environ['HF_TOKEN'] = token
os.environ['HUGGING_FACE_HUB_TOKEN'] = token  # vLLM reads this one
login(token=token, add_to_git_credential=False)
print('logged in as:', __import__('huggingface_hub').whoami()['name'])

In [ ]:
# (Optional) Mount Drive to back up generated JSONs
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    import os
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
except ImportError:
    pass

In [ ]:
# Config — Gemma 3 12B (~12B active, fits on T4/L4/A100 in bf16)
MODEL = 'google/gemma-3-12b-it'
STRUCTURAL_N = 200                     # per category (4 categories -> 400 total)
CONFLICT_N = 600
VLLM_MAX_LEN = 4096
VLLM_GPU_MEM = 0.90

from pathlib import Path
DATA_DIR = Path('/content/Mech_spoof/data')
assert DATA_DIR.exists(), f'Seed data missing at {DATA_DIR}'

In [ ]:
# Expand structural instructions (format / persona / behavioral / style)
from mech_spoof.cli.build_datasets import expand_structural

expand_structural(
    target_per_category=STRUCTURAL_N,
    backend='vllm',
    model=MODEL,
    data_dir=DATA_DIR,
    engine_kwargs={
        'max_model_len': VLLM_MAX_LEN,
        'gpu_memory_utilization': VLLM_GPU_MEM,
    },
)

In [ ]:
# Expand conflict pairs (language / length / format / topic / name / tone / string)
from mech_spoof.cli.build_datasets import expand_conflicts

expand_conflicts(
    target_count=CONFLICT_N,
    backend='vllm',
    model=MODEL,
    data_dir=DATA_DIR,
    engine_kwargs={
        'max_model_len': VLLM_MAX_LEN,
        'gpu_memory_utilization': VLLM_GPU_MEM,
    },
)

In [ ]:
# Cache AdvBench + write matched harmless set
from mech_spoof.datasets.advbench import load_advbench
from mech_spoof.datasets.harmless import write_harmless_json

print('advbench:', len(load_advbench()))
print('harmless:', write_harmless_json(DATA_DIR))

In [ ]:
# Release vLLM GPU memory
from mech_spoof.datasets._vllm_gen import free_llm
free_llm()

In [ ]:
# Backup generated JSONs to Drive (optional) + quick peek
import json, shutil
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/mech_spoof_data')
if Path('/content/drive/MyDrive').exists():
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    for src in DATA_DIR.rglob('*.json'):
        dst = DRIVE_DATA / src.relative_to(DATA_DIR)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print('backed up to', DRIVE_DATA)

for p in sorted(DATA_DIR.rglob('*.json')):
    n = len(json.loads(p.read_text())) if p.stat().st_size > 0 else 0
    print(f'{p.relative_to(DATA_DIR)}  ({n} items)')

## Committing the datasets

After the notebook finishes you'll have the full JSONs at `/content/Mech_spoof/data/`. To use them in every future run, commit them back to the repo:

```bash
%cd /content/Mech_spoof
!git add data/
!git -c user.email=you@example.com -c user.name=you commit -m 'Expand datasets via vLLM'
!git push  # needs a PAT set up via gh auth login or similar
```

Or just download the JSONs locally and commit from your workstation.
